# AllSides News Article Scraper

This notebook collects political news articles from archived **AllSides** pages for downstream quantitative and NLP analysis.

The scraper uses **Playwright** to render archived pages through the Wayback Machine and **BeautifulSoup** to parse article metadata and text.

### Workflow

1. Load an archived AllSides balanced-news page
2. Extract article links and publication names
3. Follow each AllSides article page to the original publisher link
4. Scrape the linked article text
5. Combine article text with its publication source
6. Export the resulting dataset as a CSV

The notebook is intentionally sequential so the scraping process is easy to inspect and debug.


## 1. Setup

In [ ]:
!pip install -q playwright
!playwright install
!playwright install-deps


In [ ]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import time
import numpy as np
import pandas as pd


## 2. Browser scraping function

Playwright launches a headless Chromium browser and loads the requested page before returning its rendered HTML to BeautifulSoup.

A longer navigation timeout is useful for archived Wayback Machine pages, which can load more slowly than live webpages.


In [ ]:
async def scrape(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        context = await browser.new_context(
            user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/122 Safari/537.36"
        )

        page = await context.new_page()
        page.set_default_navigation_timeout(60000)
        page.set_default_timeout(60000)

        await page.goto(
            url,
            timeout=60000,
            wait_until="domcontentloaded"
        )

        html = await page.content()

        await browser.close()

    return BeautifulSoup(html, "html.parser")

## 3. Load the archived AllSides page

The Wayback Machine snapshot preserves an earlier version of the AllSides balanced-news page used for article collection.


In [ ]:
soup = await scrape(
    "https://web.archive.org/web/20220322090124/https://www.allsides.com/unbiased-balanced-news"
)


## 4. Extract article URLs and publication names

Each article is stored as a tuple containing its AllSides URL and its publication/source name.


In [ ]:
articles = []

for s in soup.select(".news-trio .news-item"):
    articles.append(
        (
            s.select_one("a")["href"],
            s.select_one("a.source-area div.news-source span").text
        )
    )

print(f"Collected {len(articles)} article references.")
articles[:5]


## 5. Resolve publisher links

Each AllSides article page contains a link to the original publisher. The loop follows those pages and extracts the destination URL.

Random delays reduce the request rate while iterating through archived pages.


In [ ]:
article_links = []
rng = np.random.default_rng()

for index, l in enumerate(articles):
    try:
        article_soup = await scrape(l[0])

        article_links.append(
            article_soup
            .select_one(".read-more-story")
            .select_one("a")["href"]
        )

        print(index)

        rand_float = rng.uniform(12, 16)
        time.sleep(rand_float)

    except Exception as e:
        print(index, e)


In [ ]:
article_links

## 6. Scrape article text

The resolved publisher URLs are loaded one at a time and the visible page text is stored for later analysis.


In [ ]:
article_content = []

for index, l in enumerate(article_links):

    rand_float = rng.uniform(12, 16)
    time.sleep(rand_float)

    try:
        soup = await scrape(l)
        article_content.append(soup.text)
        print(index)

    except Exception as e:
        print(index, e)


## 7. Construct the dataset

The scraped article text is paired with its publication source and converted into a pandas DataFrame.


In [ ]:
finished_scrape = [
    (a[1], b)
    for a, b in zip(articles, article_content)
]

df = pd.DataFrame(
    finished_scrape,
    columns=["source", "article_text"]
)

df.head()


## 8. Export

Save the resulting dataset locally as a CSV. The output can then be used for ranking, labeling, or NLP experiments.


In [ ]:
df.to_csv("allsides_articles.csv", index=False)

print(f"Saved {len(df)} articles to allsides_articles.csv")


## Notes

- Archived webpages can be slow or occasionally unavailable, so individual requests may fail.
- Publisher websites use different HTML structures; this notebook collects full rendered page text rather than relying on publisher-specific selectors.
- The scraping delay is intentionally conservative to avoid sending rapid repeated requests.
